# 27 · Rate ratio with an interval against statsmodels

**Question.** When Fieldwork says the cholera death rate among houses supplied by the
Southwark and Vauxhall Company was **8.40 times** the rate among Lambeth's, with a 95%
interval of **6.84 to 10.32**, does an independent implementation get the same three numbers?

This is the claim the South London example exists to make. An epidemiologist will check it
against their own calculation before they trust anything else on the screen, and until this
check it rested on Fieldwork's own tests alone.

**The estimator matters.** A rate ratio is not a risk ratio. Fieldwork states
`ln(RR) ± z·sqrt(1/a + 1/b)` over event counts; a risk ratio over the same table uses
`sqrt(1/a - 1/n1 + 1/b - 1/n2)` and gives different bounds. Comparing against the wrong one
would have produced a confident disagreement about nothing.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["statsmodels", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

`wald-log` is the method Fieldwork declares. `score` is carried beside it as a third
opinion — a different statistic, its variance estimated under the null rather than from the
observed rates — so what the declared choice costs is visible rather than hidden.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_27_rate_ratio import run

report = run(Path.cwd() / "fixtures" / "rate-ratio-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    f, s = c["fieldwork"], c["scoreMethod"]
    print(f"{c['note']}: {f['ratio']:.4f} [{f['lower']:.4f}, {f['upper']:.4f}] "
          f"· score [{s['lower']:.4f}, {s['upper']:.4f}]")


## What this check found

On its first run every interval missed by more than 1e-6 while every ratio agreed exactly —
the signature of a rounded constant rather than a wrong formula. Fieldwork hard-coded the
95% normal quantile as **1.9600** where it is **1.959963984540054**, moving Snow's per-house
lower bound from 6.8409 to 6.8410.

Immaterial at two decimals, and exactly what an independent check exists to find before a
reader does. It was fixed and the fixture regenerated; the cell below is what caught it.


In [ ]:
from scipy.stats import norm
print("scipy  :", norm.ppf(0.975))
print("used   :", report["zQuantile"]["fieldwork"], "— agrees:", report["zQuantile"]["agrees"])


## What this cannot establish

**That the comparison is fair.** The two companies' customers were not randomised and no
interval speaks to confounding. Snow's own argument rests on the intermixing of supplies
within the same streets, which is a fact about the setting rather than about the data.

**That the log-normal interval is adequate at small counts.** The Lambeth-only group has 18
events. An exact conditional interval would be the better comparison there, and statsmodels
documents that method as not yet implemented — recorded rather than worked around, because a
hand-rolled exact interval would be this author's arithmetic checking this author's.

**What the right denominator is.** Houses and people give different ratios over the same
outbreak — 8.40 and 5.35 — and choosing between them is the practitioner's.
